In [1]:
import os, ast

def find(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

SRC = find("10_train_student_d32.py")
s = open(SRC).read()

def rep(a, b):
    global s
    assert s.count(a) == 1, f"pola tidak ditemukan tepat satu kali: {a[:60]!r}"
    s = s.replace(a, b)

# 1) opsi baru
rep('    accum=1,', '    side_csv="",        # knee_side.csv: citra koronal & aksial lutut kiri dicerminkan\n    accum=1,')
rep('    ap.add_argument("--accum", type=int, default=CFG["accum"])',
    '    ap.add_argument("--accum", type=int, default=CFG["accum"])\n    ap.add_argument("--side_csv", default=CFG["side_csv"])')
# 2) dataset: cerminkan view koronal & aksial untuk lutut kiri (dua baris diganti terpisah)
rep('    def __init__(self, uids, ymat, vol_dir, train):',
    '    def __init__(self, uids, ymat, vol_dir, train, flip=None, flip_idx=()):')
rep('        self.uids, self.y, self.vol_dir, self.train = uids, ymat, vol_dir, train',
    '        self.uids, self.y, self.vol_dir, self.train = uids, ymat, vol_dir, train\n'
    '        self.flip, self.flip_idx = (flip or {}), list(flip_idx)')
rep('        vmask = torch.from_numpy(d["mask"]).float()                 # (V,)',
    '        vmask = torch.from_numpy(d["mask"]).float()                 # (V,)\n'
    '        if self.flip_idx and self.flip.get(self.uids[i], False):\n'
    '            vol[self.flip_idx] = vol[self.flip_idx].flip(-1)          # lutut kiri -> cermin kiri-kanan')
# 3) peta sisi lutut dan indeks view yang dicerminkan
rep('    have = set(vol_map)',
    '    have = set(vol_map)\n'
    '    FLIP, FLIP_IDX = {}, []\n'
    '    if CFG["side_csv"]:\n'
    '        sd = pd.read_csv(find_file(CFG["side_csv"]))\n'
    '        FLIP = dict(zip(sd["StudyInstanceUID"], sd["side"] == "L"))\n'
    '        u0 = next(iter(vol_map)); nv = int(np.load(os.path.join(vol_map[u0], f"{u0}.npz"))["mask"].shape[0])\n'
    '        FLIP_IDX = {3: [1, 2], 5: [1, 2, 4]}[nv]          # v1: sag, cor, ax | v2/v3: sag, cor, ax, sag, cor\n'
    '        print(f"Normalisasi sisi lutut: {sum(FLIP.get(u, False) for u in have)} studi kiri dicerminkan | view {FLIP_IDX}")')
rep('                    vol_map, train)', '                    vol_map, train, FLIP, FLIP_IDX)')

ast.parse(s)                                     # cek sintaks
open("/kaggle/working/11_train_student_lat.py", "w").write(s)
print("Script dibuat dari:", SRC)
print("Cek baris baru:")
for line in s.splitlines():
    if "flip" in line.lower() and ("def __init__" in line or "self.flip" in line or "FLIP_IDX =" in line or ".flip(-1)" in line):
        print("  ", line.strip())

Script dibuat dari: /kaggle/input/notebooks/kellenvalerie/rsna-knee-14-student-d32/10_train_student_d32.py
Cek baris baru:
   def __init__(self, uids, ymat, vol_dir, train, flip=None, flip_idx=()):
   self.flip, self.flip_idx = (flip or {}), list(flip_idx)
   if self.flip_idx and self.flip.get(self.uids[i], False):
   vol[self.flip_idx] = vol[self.flip_idx].flip(-1)          # lutut kiri -> cermin kiri-kanan
   FLIP, FLIP_IDX = {}, []
   FLIP_IDX = {3: [1, 2], 5: [1, 2, 4]}[nv]          # v1: sag, cor, ax | v2/v3: sag, cor, ax, sag, cor


In [2]:
%%bash
cd /kaggle/working
BASE="--alpha 0.5 --oof oof_effb0_kf2.csv --side_csv knee_side.csv"
DINO="--backbone vit_small_patch14_dinov2.lvd142m --lr_backbone 1e-4 --llrd 0.8"

# GPU 0: DINOv2 data v2 (5 view), lalu EffB0 data v2
( CUDA_VISIBLE_DEVICES=0 python -u 11_train_student_lat.py $BASE $DINO \
    --meta _meta_v2.csv --batch 2 --epochs 12 --tag dinov2s_student_v3_lat \
    > log_dinov2s_student_v3_lat.txt 2>&1
  CUDA_VISIBLE_DEVICES=0 python -u 11_train_student_lat.py $BASE \
    --backbone efficientnet_b0 --meta _meta_v2.csv --batch 2 --tag effb0_student_v3_lat \
    > log_effb0_student_v3_lat.txt 2>&1 ) &

# GPU 1: DINOv2 32 irisan
CUDA_VISIBLE_DEVICES=1 python -u 11_train_student_lat.py $BASE $DINO \
    --meta _meta_v3 --batch 1 --accum 2 --epochs 10 --tag dinov2s_student_d32_lat \
    > log_dinov2s_student_d32_lat.txt 2>&1 &
wait
for t in dinov2s_student_v3_lat effb0_student_v3_lat dinov2s_student_d32_lat; do
  echo "===== $t ====="; tail -n 16 log_$t.txt
done
grep -H "Normalisasi\|Train \|Jumlah irisan\|Jumlah view\|Error\|error" log_*.txt

===== dinov2s_student_v3_lat =====
Epoch 11: loss 0.3853 | val AUC 0.8720 | val-ref AUC 0.8892 | gold AUC 0.8529 | 22.0 menit

=== [dinov2s_student_v3_lat] Checkpoint terbaik (val-ref AUC 0.8892) pada label emas ===
ACL                AUC=0.890
MCL                AUC=0.848
Medial Meniscus    AUC=0.888
Lateral Meniscus   AUC=0.717
Medial OA          AUC=0.972
Lateral OA         AUC=0.807
PF OA              AUC=0.792
Effusion           AUC=0.888
Synovitis          AUC=0.736
Baker's            AUC=0.973
Contusion          AUC=0.865
Fracture           AUC=0.860
Mean AUC gold: 0.8529
===== effb0_student_v3_lat =====
Epoch 5: loss 0.3987 | val AUC 0.8401 | val-ref AUC 0.8587 | gold AUC 0.8318 | 14.5 menit

=== [effb0_student_v3_lat] Checkpoint terbaik (val-ref AUC 0.8587) pada label emas ===
ACL                AUC=0.795
MCL                AUC=0.748
Medial Meniscus    AUC=0.816
Lateral Meniscus   AUC=0.789
Medial OA          AUC=0.977
Lateral OA         AUC=0.878
PF OA              AUC=0.781


In [3]:
import pandas as pd
for tag in ["dinov2s_student_v3_lat", "effb0_student_v3_lat", "dinov2s_student_d32_lat"]:
    print(f"\n===== {tag} =====")
    try:
        print(pd.read_csv(f"/kaggle/working/train_log_{tag}.csv").round(4).to_string(index=False))
    except FileNotFoundError:
        print("log belum ada")


===== dinov2s_student_v3_lat =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5457   0.7283       0.7440    0.7648 22.0445
     1 0.4904   0.7752       0.7933    0.7606 22.0136
     2 0.4714   0.8059       0.8238    0.8064 22.0007
     3 0.4528   0.8249       0.8411    0.8411 21.9649
     4 0.4396   0.8427       0.8601    0.8392 21.9761
     5 0.4277   0.8510       0.8681    0.8511 21.9503
     6 0.4176   0.8599       0.8771    0.8491 21.9462
     7 0.4074   0.8656       0.8825    0.8531 21.9137
     8 0.3994   0.8699       0.8856    0.8475 21.9725
     9 0.3924   0.8708       0.8880    0.8518 21.9942
    10 0.3879   0.8720       0.8892    0.8524 22.0362
    11 0.3853   0.8720       0.8892    0.8529 21.9915

===== effb0_student_v3_lat =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.4971   0.7765       0.7936    0.7855 16.4483
     1 0.4449   0.8154       0.8332    0.8035 14.5752
     2 0.4260   0.8319       0.8527    0.8212 14.5531
     3 0.412